# 00 - Setup and First Checks

This notebook helps you verify that your machine is ready for Day 1. 
Use it together with `INSTRUCTIONS.MD`.

## Learning goals

- Confirm that `git`, Python, and `jupyter` are available.
- Know whether you are using the preferred conda route or the fallback venv/pip route.
- Check that the notebook is running in the expected Python environment.
- Test whether the core course libraries import correctly.
- Confirm that the bundled BAFU workbook is present.
- Run a tiny Brightway LCA and confirm that changing one input changes the score.
- Know how to launch Jupyter Lab and switch to the correct kernel.


## Background reference

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236


## 1) Environment snapshot


In [1]:
# we first import the libraries we will use
import os
import platform
import sys
from pathlib import Path

print('Python version:', sys.version.split()[0])
print('Python executable:', sys.executable)
print('Platform:', platform.platform())
print('Current folder:', Path.cwd())
normalized = sys.executable.replace('\\', '/').lower()
print('Looks like bw env:', 'envs/bw' in normalized or '.venv-bw' in normalized or '/bw/' in normalized)


Python version: 3.11.15
Python executable: /opt/homebrew/Caskroom/miniforge/base/envs/bw/bin/python
Platform: macOS-26.5.1-arm64-arm-64bit
Current folder: /Users/romain/GitHub/aalborg-regionalized-lcia/tutorials/00_SETUP
Looks like bw env: True


## 2) Command-line tools

These tools should be available from the shell you use for the course.

In a terminal, the key commands are:

```bash
git --version
python --version
jupyter lab

# Preferred conda/Miniforge route:
conda --version
conda env list
```


In [3]:
# but we can also check their presence from Python
import shutil

for tool in ['git', 'python', 'jupyter']:
    print(f'{tool}:', shutil.which(tool) or 'NOT FOUND on PATH')

print('conda path:', shutil.which('conda') or 'NOT FOUND on PATH')


git: /usr/bin/git
python: /opt/homebrew/Caskroom/miniforge/base/envs/bw/bin/python
jupyter: /opt/homebrew/Caskroom/miniforge/base/envs/bw/bin/jupyter
conda path: /opt/homebrew/Caskroom/miniforge/base/condabin/conda


## 3) Import checks in the current kernel

The goal is to verify whether the notebook kernel (`bw`) can see the main course packages.

In [4]:
import importlib

modules = ['bw2data', 'bw2calc', 'bw2io', 'edges', 'clips']

for name in modules:
    try:
        module = importlib.import_module(name)
        version = getattr(module, '__version__', 'unknown')
        print(f'{name}: OK | version={version}')
    except Exception as exc:
        print(f'{name}: FAIL | {type(exc).__name__}: {exc}')


/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


bw2data: OK | version=4.7
bw2calc: OK | version=2.4.0
bw2io: OK | version=0.9.17
edges: OK | version=1.3.1
clips: OK | version=1.0.6


## 5) Check the bundled BAFU workbook

The Day 1 import notebook uses the workbook already shipped with this repository:

- `data/lci-bafu.xlsx`


In [5]:
from datetime import datetime
from pathlib import Path

bafu_path = Path('../../data/lci-bafu.xlsx')
print('Expected path:', bafu_path.resolve())
print('Found:', bafu_path.exists())


Expected path: /Users/romain/GitHub/aalborg-regionalized-lcia/data/lci-bafu.xlsx
Found: True


## 6) Readiness mini-exercise: run, predict, change

This exercise proves that the active kernel can perform a complete LCI and LCIA—not only import the packages. Its purpose is also to get you acquainted with the LCA workflow we will use on DAY 1.

> **Why this code exists**
>
> - **Input:** one service uses a configurable amount of grid electricity; grid electricity emits `0.5 kg CO2` per kWh.
> - **Purpose:** verify the full path from model data to inventory to characterized score.
> - **Produces:** one climate-change score in `kg CO2-eq` and an automatic pass/fail check.
> - **If it fails:** use [TROUBLESHOOTING.md](../../TROUBLESHOOTING.md), starting with the kernel and core-import sections.


### 6.1 Set one model input and predict the result

Before running the next cells, predict the score: `2.0 kWh × 0.5 kg CO2/kWh = 1.0 kg CO2-eq`.


In [6]:
electricity_kwh = 2.0
grid_co2_per_kwh = 0.5

print(f'Model input: {electricity_kwh} kWh of electricity per service')
print(f'Expected score: {electricity_kwh * grid_co2_per_kwh:.2f} kg CO2-eq')


Model input: 2.0 kWh of electricity per service
Expected score: 1.00 kg CO2-eq


### 6.2 Build the tiny model

The database-writing code is intentionally visible. You do not need to memorise it before the course; follow how the parameter above becomes a technosphere exchange below. This cell is safe to rerun.

Do not worry if you do not understand all of what the code does, as we will go into details about it on DAY1.

In [9]:
import bw2calc as bc # bw2calc is about doing LCA calculations
import bw2data as bd # bw2data is about creating, deleting and modifying LCA projects and databases

# let's define some names
readiness_project = 'aalborg-check' # a project name
biosphere_name = 'biosphere-check' # a biosphere database name
foreground_name = 'foreground-check' # a technosphere databas ename
method_key = ('Aalborg check', 'climate change') # an LCIA method name

# let's create the project and enter it
bd.projects.set_current(readiness_project)

# let's create the biosphere database and write a fossil CO2 elementary flow in it
bd.Database(biosphere_name).write({
    (biosphere_name, 'co2'): {
        'name': 'Carbon dioxide, fossil',
        'unit': 'kilogram',
        'type': 'emission',
    },
})

# let's create a technosphere database with two activities in it
# pay attention to the structure and try to understand how it works
# but we will go into the details of it on DAY 1
bd.Database(foreground_name).write({
    (foreground_name, 'grid-electricity'): {
        'name': 'Grid electricity',
        'reference product': 'electricity',
        'unit': 'kilowatt hour',
        'location': 'DK',
        'exchanges': [
            {'input': (foreground_name, 'grid-electricity'), 'amount': 1.0, 'type': 'production'},
            {'input': (biosphere_name, 'co2'), 'amount': grid_co2_per_kwh, 'type': 'biosphere'},
        ],
    },
    (foreground_name, 'service'): {
        'name': 'My service',
        'reference product': 'service',
        'unit': 'unit',
        'location': 'DK',
        'exchanges': [
            {'input': (foreground_name, 'service'), 'amount': 1.0, 'type': 'production'},
            {'input': (foreground_name, 'grid-electricity'), 'amount': electricity_kwh, 'type': 'technosphere'},
        ],
    },
})

# if the LCIA method does not exist, we create it
if method_key not in bd.methods:
    bd.Method(method_key).register(unit='kg CO2-eq')
# and we add a characterisation factor into it for fossil CO2
bd.Method(method_key).write([((biosphere_name, 'co2'), 1.0)])

print('Current project:', bd.projects.current)
print('Tiny databases:', biosphere_name, 'and', foreground_name)


100%|██████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 18893.26it/s]

09:03:19+0200 [info     ] Vacuuming database            



100%|██████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 34952.53it/s]

09:03:19+0200 [info     ] Vacuuming database            
Current project: aalborg-check
Tiny databases: biosphere-check and foreground-check


### 6.3 Calculate and check the score

The functional unit is exactly one of `My service`. Brightway solves its supply chain, builds the inventory, applies the one characterisation factor, and compares the result with the value predicted above.


In [12]:
# we select the functional unit
service = bd.get_activity((foreground_name, 'service'))

# we instantiate the LCA object, give it the functional unit and the LCIA method
lca = bc.LCA({service: 1}, method=method_key)

# we solve the system
lca.lci()

# we characterise the inventory
lca.lcia()

expected_score = electricity_kwh * grid_co2_per_kwh
print(f'Calculated score: {lca.score:.2f} kg CO2-eq')
print(f'Expected score:   {expected_score:.2f} kg CO2-eq')

assert abs(lca.score - expected_score) < 1e-9, 'The calculated score does not match the model inputs.'
print('READINESS CHECK: PASS')


Calculated score: 1.00 kg CO2-eq
Expected score:   1.00 kg CO2-eq
READINESS CHECK: PASS


## 7) Make one change yourself

1. Return to the input cell and change `electricity_kwh = 2.0` to `electricity_kwh = 3.0`.
2. Before running anything, predict the new result.
3. Rerun the input, model-building, and calculation cells in order.
4. Confirm that the score changes from `1.0` to `1.5 kg CO2-eq` and that the final line still says `READINESS CHECK: PASS`.

If the printed input changes but the score does not, a downstream cell was not rerun. This dependency between an edited input and all later cells is an important Jupyter habit for the course.


## Recap

After this notebook, you should be able to confirm:

- which Python executable your notebook uses
- whether the core course libraries are importable
- whether the BAFU database is present
- whether a complete Brightway LCI and LCIA runs successfully
- whether changing one model input changes the result as predicted
- whether Jupyter is using the right kernel for the course
